# 5c — Plot Flood Scenarios Accessibility

## Purpose
Visualise the basin-scenario accessibility results computed by script 5b: increased travel times from settlements to emergency services (hospitals, police, fire stations), from industrial areas to border crossings, and from agricultural areas to road borders, ports, and rail terminals. The exposed-edge impact layers are saved to parquet + ArcGIS File Geodatabase + Excel in EPSG:6316.

## Inputs
| File | Description |
|------|-------------|
| `intermediate_results/SRB_flood_statistics_per_Basin_basins_scenario.csv` | Flood statistics per basin (from 5b) |
| `input_files/hybas_eu_lev09_v1c.shp` | HydroBASINS level-9 basin polygons |
| `input_files/base_network_SRB_basins.parquet` | Basin-scenario base road network |
| `accessibility_analysis/{healthcare,police,fire,factory}_criticality_results/save_new_results_SRB_basins.pkl` | Per-basin disruption results per service (from 5b) |
| `accessibility_analysis/allagri_criticality_results/save_new_results_SRB_basins.pkl` | Per-basin agriculture disruption results (from 5b) |

## Outputs
| File | Description |
|------|-------------|
| `figures/basin_water_depths.png` | Mean water depth per basin |
| `figures/hospital_criticality.png` | Increased travel time to hospitals |
| `figures/police_criticality.png` | Increased travel time to police stations |
| `figures/fire_criticality.png` | Increased travel time to fire stations |
| `figures/factory_criticality.png` | Increased travel time from industrial areas to borders |
| `figures/SRB_agri_criticality_avg_3x1.png` | Agriculture: average increased travel time (borders / ports / rail) |
| `figures/SRB_agri_criticality_nearest_3x1.png` | Agriculture: increased travel time to nearest sink |
| `figures/criticality_2x2.png` | Combined 2×2 figure (hospitals, factories, police, fire) |
| `intermediate_results/parquet/local_accessibility/{hospital,factory,police,fire,road,rail,port}_impacts.parquet` | Impact layers (EPSG:6316) |
| `intermediate_results/parquet/local_accessibility/{...}_impacts.xlsx` | Excel attribute tables |
| `intermediate_results/database/local_accessibility/local_accessibility.gdb` | GDB layers for all impact layers |

## Key Processing Steps
1. **Basin water depths** — merge per-basin flood statistics with HydroBASINS geometries; plot depth classes
2. **Base network** — load basin-scenario road network; keep the giant connected component (EPSG:3857)
3. **Service criticality** — per service (hospitals, police, fire, factories): assign each basin's mean positive travel-time delta to its removed edges; keep exposed edges with ≥ 10 min impact; bin into delay classes
4. **Agriculture criticality** — same logic per sink type (road borders, ports, rail terminals) for the average and nearest-sink delta columns
5. **Figures** — single map per service, 3×1 agriculture figures (average and nearest), combined 2×2 figure
6. **Save** — all seven impact layers to parquet + GDB + Excel (EPSG:6316)

<div style="
    border-left: 6px solid #1f6feb;
    padding: 0.7em 1em;
    background: #f0f7ff;
    border-radius: 4px;
    margin: 1em 0;
    font-size: 1.05em;
    color: #0a0a0a;
">
  <strong>📝 Note:</strong> This notebook requires the results computed by script 5b_Run_Flood_Scenarios_Accessibility.py to run
</div>

In [ ]:
import sys
import warnings
from pathlib import Path

BASE_DIR = Path(r"C:\Users\yma794\Documents\Serbia\analysis rerun 23_3\Criticality-Analysis-Roads-Serbia")
sys.path.append(str(BASE_DIR))

from utils.criticality_functions import (
    calculate_agri_criticality,
    calculate_service_criticality,
    load_base_network,
    load_basins,
    plot_agri_criticality_3x1,
    plot_basin_water_depths,
    plot_criticality_2x2,
    plot_service_criticality,
    save_impact_layers,
)

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=RuntimeWarning)

In [ ]:
data_path = BASE_DIR / "input_files"
figure_path = BASE_DIR / "figures"
intermediate_results_path = BASE_DIR / "intermediate_results"
accessibility_analysis_path = BASE_DIR / "accessibility_analysis"
parquet_dir = intermediate_results_path / "parquet" / "local_accessibility"
gdb_path = intermediate_results_path / "database" / "local_accessibility" / "local_accessibility.gdb"
output_crs = "EPSG:6316"

In [ ]:
# Basin water depths
basins = load_basins(
    intermediate_results_path / "SRB_flood_statistics_per_Basin_basins_scenario.csv",
    data_path / "hybas_eu_lev09_v1c.shp",
)
plot_basin_water_depths(basins, figure_path)

In [ ]:
# Base network (giant connected component, EPSG:3857)
base_network = load_base_network(data_path / "base_network_SRB_basins.parquet")

## Emergency Services and Factories

In [ ]:
hospital_exposed_edges = calculate_service_criticality(
    accessibility_analysis_path / "healthcare_criticality_results" / "save_new_results_SRB_basins.pkl",
    base_network,
)
plot_service_criticality(hospital_exposed_edges, base_network, figure_path, "hospital_criticality.png")

In [ ]:
police_exposed_edges = calculate_service_criticality(
    accessibility_analysis_path / "police_criticality_results" / "save_new_results_SRB_basins.pkl",
    base_network,
)
plot_service_criticality(police_exposed_edges, base_network, figure_path, "police_criticality.png")

In [ ]:
fire_exposed_edges = calculate_service_criticality(
    accessibility_analysis_path / "fire_criticality_results" / "save_new_results_SRB_basins.pkl",
    base_network,
)
plot_service_criticality(fire_exposed_edges, base_network, figure_path, "fire_criticality.png")

In [ ]:
factory_exposed_edges = calculate_service_criticality(
    accessibility_analysis_path / "factory_criticality_results" / "save_new_results_SRB_basins.pkl",
    base_network,
)
plot_service_criticality(factory_exposed_edges, base_network, figure_path, "factory_criticality.png")

## Agriculture Criticality (Road Borders / Ports / Rail Terminals)

In [ ]:
agri_results_path = (
    accessibility_analysis_path / "allagri_criticality_results" / "save_new_results_SRB_basins.pkl"
)

# Average increased travel time over all sinks of each type
agri_avg = calculate_agri_criticality(agri_results_path, base_network, delta_prefix="delta_avg")
plot_agri_criticality_3x1(
    agri_avg, base_network, figure_path,
    file_name="SRB_agri_criticality_avg_3x1.png",
    legend_title="Average Increased Travel Time",
)

In [ ]:
# Increased travel time to the nearest sink of each type
agri_nearest = calculate_agri_criticality(agri_results_path, base_network, delta_prefix="delta_nearest")
plot_agri_criticality_3x1(
    agri_nearest, base_network, figure_path,
    file_name="SRB_agri_criticality_nearest_3x1.png",
    legend_title="Increased Travel Time To Nearest",
)

## Combined Criticality Figure

In [ ]:
plot_criticality_2x2(
    hospital_exposed_edges, factory_exposed_edges,
    police_exposed_edges, fire_exposed_edges,
    base_network, figure_path,
)

### Save all results for the combined criticality assessment

In [ ]:
impact_layers = {
    "hospital_impacts": hospital_exposed_edges,
    "factory_impacts": factory_exposed_edges,
    "police_impacts": police_exposed_edges,
    "fire_impacts": fire_exposed_edges,
    "road_impacts": agri_nearest["road"],
    "rail_impacts": agri_nearest["rail"],
    "port_impacts": agri_nearest["port"],
}
save_impact_layers(impact_layers, parquet_dir, gdb_path, output_crs)